In [1]:
# Libraries
import caveclient
import subprocess
import json
import numpy as np
import matplotlib.pyplot as plt
# Setup caveclient
client = caveclient.CAVEclient("wclee_mouse_spinalcord_cltmr", ) #server_address='https://global.daf-apis.com')

In [2]:
cell_tag = client.materialize.query_table('sensory_neuron_label')

In [4]:
import os
import re
import numpy as np
import pandas as pd
import trimesh
import cloudvolume as cv
import open3d as o3d

# --- config ---
OUT_BASE = '/Users/wangchu/mesh_analysis/sensory_mesh_downsample'

DOWNSAMPLE_TIMES   = 6        # ← set to 5 for “5×”
MIN_TARGET_FACES   = 2000
APPLY_DOWNSAMPLE   = True
CHUNK_SIZE         = 64

# Resume/overwrite controls
SKIP_EXISTING      = True     # skip if <rid>.ply already exists and is non-empty
FORCE_OVERWRITE    = False    # set True to ignore existing and rewrite everything

vol = cv.CloudVolume(
    'graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr',
    secrets='REPLACE_WITH_YOUR_CAVE_TOKEN',
    use_https=True,
    progress=False
)

os.makedirs(OUT_BASE, exist_ok=True)

def sanitize_folder(name: str, maxlen: int = 120) -> str:
    s = str(name).strip()
    s = re.sub(r'[\\/:*?"<>|]+', '_', s)
    s = re.sub(r'\s+', '_', s)
    return s[:maxlen] if s else "untagged"

def decimate_o3d(tri: trimesh.Trimesh, factor: int = DOWNSAMPLE_TIMES) -> trimesh.Trimesh:
    if tri.faces is None or tri.faces.size == 0:
        return tri
    original_faces = int(tri.faces.shape[0])
    target = max(MIN_TARGET_FACES, original_faces // max(1, factor))
    if target >= original_faces:
        return tri

    m = o3d.geometry.TriangleMesh()
    m.vertices = o3d.utility.Vector3dVector(tri.vertices.astype(np.float64, copy=False))
    m.triangles = o3d.utility.Vector3iVector(tri.faces.astype(np.int32, copy=False))
    if not m.has_vertex_normals():
        m.compute_vertex_normals()

    ms = m.simplify_quadric_decimation(target_number_of_triangles=int(target))
    ms.remove_duplicated_vertices()
    ms.remove_duplicated_triangles()
    ms.remove_degenerate_triangles()
    ms.remove_non_manifold_edges()

    v = np.asarray(ms.vertices); f = np.asarray(ms.triangles)
    if v.size == 0 or f.size == 0:
        print("[warn] Decimator returned empty mesh; using original.")
        return tri

    out = trimesh.Trimesh(vertices=v, faces=f, process=False)
    out.remove_unreferenced_vertices()
    print(f"[info] Decimated: {original_faces} → {out.faces.shape[0]} faces.")
    return out

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def parse_existing_ids(folder: str):
    """Return a set of root_ids that already have a non-empty .ply in folder."""
    ids = set()
    if not os.path.isdir(folder):
        return ids
    for fn in os.listdir(folder):
        if not fn.lower().endswith('.ply'):
            continue
        try:
            rid_str = os.path.splitext(fn)[0]
            rid = int(rid_str)
            fp = os.path.join(folder, fn)
            if os.path.getsize(fp) > 0:
                ids.add(rid)
        except Exception:
            continue
    return ids

def atomic_write_ply(tri: trimesh.Trimesh, out_path: str):
    """Write PLY atomically via .part then rename."""
    tmp = out_path + ".part"
    ply_bytes = trimesh.exchange.ply.export_ply(tri)
    with open(tmp, 'wb') as f:
        f.write(ply_bytes)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, out_path)  # atomic on POSIX

# --- validate input DataFrame (case-insensitive) ---
required = {'pt_root_id', 'tag'}
lower_map = {c.lower(): c for c in cell_tag.columns}
if not required.issubset(set(lower_map.keys())):
    raise ValueError("cell_tag must have columns 'pt_root_id' and 'tag' (any case).")
cell_tag = cell_tag.rename(columns={lower_map['pt_root_id']: 'pt_root_id',
                                   lower_map['tag']: 'tag'})

df = (cell_tag.dropna(subset=['pt_root_id','tag'])
               .assign(pt_root_id=lambda d: pd.to_numeric(d['pt_root_id'],
                                                         errors='coerce'))
               .dropna(subset=['pt_root_id'])
               .assign(pt_root_id=lambda d: d['pt_root_id'].astype(np.int64)))

groups = df.groupby('tag')
total_exported = 0

for tag, g in groups:
    safe_tag = sanitize_folder(tag)
    out_dir = os.path.join(OUT_BASE, safe_tag)
    os.makedirs(out_dir, exist_ok=True)

    root_ids = sorted(g['pt_root_id'].unique().tolist())
    if not root_ids:
        print(f"[info] Tag '{tag}' has no valid root IDs. Skipping.")
        continue

    # Resume logic: filter out IDs that already have a good .ply
    if SKIP_EXISTING and not FORCE_OVERWRITE:
        existing = parse_existing_ids(out_dir)
        todo = [rid for rid in root_ids if rid not in existing]
    else:
        todo = root_ids

    print(f"\n=== Tag: {tag} | Total IDs: {len(root_ids)} | To do: {len(todo)} | Out: {out_dir} ===")
    if not todo:
        continue

    for batch in chunks(todo, CHUNK_SIZE):
        print(f"[info] Fetching {len(batch)} meshes…")
        try:
            mesh_dict = vol.mesh.get(batch)
        except Exception as e:
            print(f"[warn] CloudVolume fetch failed for batch of {len(batch)}: {e}")
            continue

        for rid in batch:
            out_path = os.path.join(out_dir, f"{rid}.ply")

            # Double-check skip at file level (handles races / prior runs)
            if SKIP_EXISTING and not FORCE_OVERWRITE and os.path.exists(out_path) and os.path.getsize(out_path) > 0:
                # already have a non-empty file
                continue

            mesh_raw = mesh_dict.get(rid)
            if mesh_raw is None:
                print(f'[warn] No mesh for {rid}. Skipping.')
                continue

            try:
                tm = trimesh.Trimesh(vertices=mesh_raw.vertices, faces=mesh_raw.faces, process=False)
                if tm.vertices.size == 0 or tm.faces.size == 0:
                    print(f'[warn] Empty geometry for {rid}. Skipping.')
                    continue
            except Exception as e:
                print(f'[error] Build trimesh failed for {rid}: {e}')
                continue

            tm_out = decimate_o3d(tm) if APPLY_DOWNSAMPLE else tm

            try:
                atomic_write_ply(tm_out, out_path)
                print(f"[ok] Saved {rid} → {out_path}  (faces: {tm.faces.shape[0]} → {tm_out.faces.shape[0]})")
                total_exported += 1
            except Exception as e:
                print(f'[error] Save failed for {rid}: {e}')
                # clean up any partial
                try:
                    if os.path.exists(out_path + ".part"):
                        os.remove(out_path + ".part")
                except Exception:
                    pass

print(f"\n[done] Exported {total_exported} meshes into '{OUT_BASE}' (grouped by tag).")



=== Tag: C-COLD | Total IDs: 25 | To do: 0 | Out: /Users/wangchu/mesh_analysis/sensory_mesh_downsample/C-COLD ===

=== Tag: ab-ltmr | Total IDs: 25 | To do: 0 | Out: /Users/wangchu/mesh_analysis/sensory_mesh_downsample/ab-ltmr ===

=== Tag: ad-htmr | Total IDs: 21 | To do: 0 | Out: /Users/wangchu/mesh_analysis/sensory_mesh_downsample/ad-htmr ===

=== Tag: adltmr | Total IDs: 18 | To do: 18 | Out: /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr ===
[info] Fetching 18 meshes…
[info] Decimated: 1311296 → 218549 faces.
[ok] Saved 720575940861406706 → /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr/720575940861406706.ply  (faces: 1311296 → 218549)
[info] Decimated: 1894855 → 315808 faces.
[ok] Saved 720575940879973250 → /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr/720575940879973250.ply  (faces: 1894855 → 315808)
[info] Decimated: 637614 → 106269 faces.
[ok] Saved 720575940881567427 → /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr/7205759

In [5]:
import os
import re
import numpy as np
import trimesh
import cloudvolume as cv
import open3d as o3d

# --- config ---
OUT_BASE = '/Users/wangchu/mesh_analysis/sensory_mesh_downsample'
TAG_NAME = 'adltmr'           # subfolder name

DOWNSAMPLE_TIMES   = 5        # ← set to 5 for “5×”
MIN_TARGET_FACES   = 2000
APPLY_DOWNSAMPLE   = True
CHUNK_SIZE         = 64

# Resume/overwrite controls
SKIP_EXISTING      = False     # skip if <rid>.ply already exists and is non-empty
FORCE_OVERWRITE    = True    # set True to ignore existing and rewrite everything

vol = cv.CloudVolume(
    'graphene://https://cave.fanc-fly.com/segmentation/table/wclee_mouse_spinalcord_cltmr',
    secrets='REPLACE_WITH_YOUR_CAVE_TOKEN',
    use_https=True,
    progress=False
)

os.makedirs(OUT_BASE, exist_ok=True)

def sanitize_folder(name: str, maxlen: int = 120) -> str:
    s = str(name).strip()
    s = re.sub(r'[\\/:*?"<>|]+', '_', s)
    s = re.sub(r'\s+', '_', s)
    return s[:maxlen] if s else "untagged"

def decimate_o3d(tri: trimesh.Trimesh, factor: int = DOWNSAMPLE_TIMES) -> trimesh.Trimesh:
    if tri.faces is None or tri.faces.size == 0:
        return tri
    original_faces = int(tri.faces.shape[0])
    target = max(MIN_TARGET_FACES, original_faces // max(1, factor))
    if target >= original_faces:
        return tri

    m = o3d.geometry.TriangleMesh()
    m.vertices = o3d.utility.Vector3dVector(tri.vertices.astype(np.float64, copy=False))
    m.triangles = o3d.utility.Vector3iVector(tri.faces.astype(np.int32, copy=False))
    if not m.has_vertex_normals():
        m.compute_vertex_normals()

    ms = m.simplify_quadric_decimation(target_number_of_triangles=int(target))
    ms.remove_duplicated_vertices()
    ms.remove_duplicated_triangles()
    ms.remove_degenerate_triangles()
    ms.remove_non_manifold_edges()

    v = np.asarray(ms.vertices); f = np.asarray(ms.triangles)
    if v.size == 0 or f.size == 0:
        print("[warn] Decimator returned empty mesh; using original.")
        return tri

    out = trimesh.Trimesh(vertices=v, faces=f, process=False)
    out.remove_unreferenced_vertices()
    print(f"[info] Decimated: {original_faces} → {out.faces.shape[0]} faces.")
    return out

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def parse_existing_ids(folder: str):
    """Return a set of root_ids that already have a non-empty .ply in folder."""
    ids = set()
    if not os.path.isdir(folder):
        return ids
    for fn in os.listdir(folder):
        if not fn.lower().endswith('.ply'):
            continue
        try:
            rid_str = os.path.splitext(fn)[0]
            rid = int(rid_str)
            fp = os.path.join(folder, fn)
            if os.path.getsize(fp) > 0:
                ids.add(rid)
        except Exception:
            continue
    return ids

def atomic_write_ply(tri: trimesh.Trimesh, out_path: str):
    """Write PLY atomically via .part then rename."""
    tmp = out_path + ".part"
    ply_bytes = trimesh.exchange.ply.export_ply(tri)
    with open(tmp, 'wb') as f:
        f.write(ply_bytes)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, out_path)  # atomic on POSIX

def export_meshes_for_root_ids(root_ids, tag=TAG_NAME):
    """
    Do exactly what the old script did, but driven by an explicit
    list of root_ids instead of a cell_tag DataFrame.
    """
    safe_tag = sanitize_folder(tag)
    out_dir = os.path.join(OUT_BASE, safe_tag)
    os.makedirs(out_dir, exist_ok=True)

    # force to list of ints
    root_ids = sorted(int(r) for r in root_ids)
    if not root_ids:
        print(f"[info] Tag '{tag}' has no valid root IDs. Skipping.")
        return

    # Resume logic: filter out IDs that already have a good .ply
    if SKIP_EXISTING and not FORCE_OVERWRITE:
        existing = parse_existing_ids(out_dir)
        todo = [rid for rid in root_ids if rid not in existing]
    else:
        todo = root_ids

    print(f"\n=== Tag: {tag} | Total IDs: {len(root_ids)} | To do: {len(todo)} | Out: {out_dir} ===")
    if not todo:
        return

    total_exported = 0

    for batch in chunks(todo, CHUNK_SIZE):
        print(f"[info] Fetching {len(batch)} meshes…")
        try:
            mesh_dict = vol.mesh.get(batch)
        except Exception as e:
            print(f"[warn] CloudVolume fetch failed for batch of {len(batch)}: {e}")
            continue

        for rid in batch:
            out_path = os.path.join(out_dir, f"{rid}.ply")

            # Double-check skip at file level (handles races / prior runs)
            if SKIP_EXISTING and not FORCE_OVERWRITE and \
               os.path.exists(out_path) and os.path.getsize(out_path) > 0:
                continue

            mesh_raw = mesh_dict.get(rid)
            if mesh_raw is None:
                print(f'[warn] No mesh for {rid}. Skipping.')
                continue

            try:
                tm = trimesh.Trimesh(vertices=mesh_raw.vertices,
                                     faces=mesh_raw.faces,
                                     process=False)
                if tm.vertices.size == 0 or tm.faces.size == 0:
                    print(f'[warn] Empty geometry for {rid}. Skipping.')
                    continue
            except Exception as e:
                print(f'[error] Build trimesh failed for {rid}: {e}')
                continue

            tm_out = decimate_o3d(tm) if APPLY_DOWNSAMPLE else tm

            try:
                atomic_write_ply(tm_out, out_path)
                print(f"[ok] Saved {rid} → {out_path}  "
                      f"(faces: {tm.faces.shape[0]} → {tm_out.faces.shape[0]})")
                total_exported += 1
            except Exception as e:
                print(f'[error] Save failed for {rid}: {e}')
                try:
                    if os.path.exists(out_path + ".part"):
                        os.remove(out_path + ".part")
                except Exception:
                    pass

    print(f"\n[done] Exported {total_exported} meshes into '{out_dir}'.")


if __name__ == "__main__":
    # Example: single root id
    ROOT_IDS = [720575940875019489]
    export_meshes_for_root_ids(ROOT_IDS, tag='adltmr')



=== Tag: adltmr | Total IDs: 1 | To do: 1 | Out: /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr ===
[info] Fetching 1 meshes…
[info] Decimated: 2871985 → 574397 faces.
[ok] Saved 720575940875019489 → /Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr/720575940875019489.ply  (faces: 2871985 → 574397)

[done] Exported 1 meshes into '/Users/wangchu/mesh_analysis/sensory_mesh_downsample/adltmr'.
